# Zambia Health Surveillance Synthetic Data Generator
This project generates synthetic weekly public health surveillance data for Zambia.

First, it builds a clean, predictable dataset structure for a chosen month and year: one row per disease, district, and ISO epidemiological week (so the rows and columns are always consistent). Then it asks an open source text generation LLM (either a local model via Ollama or a hosted model on Hugging Face) to fill in realistic counts (suspected cases, lab tests, confirmed cases, deaths) while outputting only CSV rows with no extra text. Users can inspect the raw model output inside the Gradio interface, and once it looks correct, convert it into a structured table and download it as a CSV.

The Realism guidance box is the prompt section. Users should use it to tell the model what patterns exist for their use case. Include (1) expected patterns (seasonality, outbreaks, rarity), (2) sensible ranges, and (3) constraints you care about. For example: “Assume routine district surveillance. Malaria is higher in rainy months (Nov–Apr) and lower in dry season; Cholera is usually zero but may spike in outbreaks; Measles occurs in small clusters; Anthrax is rare (mostly zeros). Keep deaths very low and never above confirmed. Typical weekly suspected cases: Malaria 5–120, Cholera 0–30 (only during spikes), Measles 0–15, Dysentery 0–40. If suspected is low, lab testing is also low; confirmed should be less than or equal to sent to lab.” Or if you want stress-testing: “Generate one outbreak district where Cholera spikes for 2–3 consecutive epi weeks, while other districts remain near zero; keep other diseases stable.”

In [ ]:
import os
import re
import csv
import io
import json
import random
from datetime import date, timedelta
from typing import Optional, List, Dict, Tuple

import pandas as pd
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from huggingface_hub import InferenceClient

In [ ]:
# Env and clients 
load_dotenv(override=True)

ollama_api_key = os.getenv("OLLAMA_API_KEY")
hugging_face_token = os.getenv("HUGGING_FACE_TOKEN")

def _clean_key(k: Optional[str]) -> Optional[str]:
    if not k:
        return None
    return k.strip().strip('"').strip("'")

ollama_api_key = _clean_key(ollama_api_key)
hugging_face_token = _clean_key(hugging_face_token)

if ollama_api_key:
    print(f"Ollama (Local) API Key exists and begins with '{ollama_api_key[:2]}'")
else:
    print("Ollama (Local) API Key not set (and this is optional)")

if hugging_face_token:
    print(f"Hugging Face access token exists and begins with '{hugging_face_token[:8]}'")
else:
    print("Hugging Face token not set (and this is optional)")

In [ ]:
# OpenAI-compatible client pointed at Ollama
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(base_url=ollama_url, api_key=ollama_api_key or "ollama")

# HF hosted inference client
hf = InferenceClient(token=hugging_face_token) if hugging_face_token else None

In [ ]:
# Provider and model options
MODEL_OPTIONS = {
    "Ollama": ["llama3.2"],
    "HuggingFace": ["Qwen/Qwen2.5-14B-Instruct", "MiniMaxAI/MiniMax-M2.1"],
}

In [ ]:
# Domain config
DISEASES = ["Malaria", "Cholera", "Measles", "Anthrax", "Dysentery"]

COLUMNS = [
    "disease", "province", "district",
    "year", "month",
    "iso_year", "epi_week", "week_start_date",
    "suspected_cases", "cases_sent_to_lab", "confirmed_cases", "deaths"
]

In [ ]:
# Full Zambia admin
ZAMBIA_ADMIN = {
    "Central": ["Chibombo","Chisamba","Chitambo","Kabwe","Kapiri Mposhi","Luano","Mkushi","Mumbwa","Ngabwe","Serenje","Shibuyunji"],
    "Copperbelt": ["Chililabombwe","Chingola","Kalulushi","Kitwe","Luanshya","Lufwanyama","Masaiti","Mpongwe","Mufulira","Ndola"],
    "Eastern": ["Chadiza","Chama","Chasefu","Chipangali","Chipata","Kasenengwa","Katete","Lumezi","Lundazi","Lusangazi","Mambwe","Nyimba","Petauke","Sinda","Vubwi"],
    "Luapula": ["Chembe","Chiengi","Chifunabuli","Chipili","Kawambwa","Lunga","Mansa","Milenge","Mwansabombwe","Mwense","Nchelenge","Samfya"],
    "Lusaka": ["Chilanga","Chongwe","Kafue","Luangwa","Lusaka","Rufunsa"],
    "Muchinga": ["Chinsali","Isoka","Kanchibiya","Lavushimanda","Mafinga","Mpika","Nakonde","Shiwang'andu"],
    "North-Western": ["Chavuma","Ikelenge","Kabompo","Kalumbila","Kasempa","Manyinga","Mufumbwe","Mushindamo","Mwinilunga","Solwezi","Zambezi"],
    "Northern": ["Chilubi","Kaputa","Kasama","Lunte","Lupososhi","Luwingu","Mbala","Mporokoso","Mpulungu","Mungwi","Nsama","Senga"],
    "Southern": ["Chikankanta","Chirundu","Choma","Gwembe","Itezhi-tezhi","Kalomo","Kazungula","Livingstone","Mazabuka","Monze","Namwala","Pemba","Siavonga","Sinazongwe","Zimba"],
    "Western": ["Kalabo","Kaoma","Limulunga","Luampa","Lukulu","Mitete","Mongu","Mulobezi","Mwandi","Nalolo","Nkeyema","Senanga","Sesheke","Shang'ombo","Sikongo","Sioma"],
}
PROVINCES = sorted(ZAMBIA_ADMIN.keys())
province_to_districts = {p: sorted(ZAMBIA_ADMIN[p]) for p in PROVINCES}
district_to_province = {d.lower(): p for p, ds in province_to_districts.items() for d in ds}

def norm_name(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"[,\s]+$", "", s)
    s = re.sub(r"\s+", " ", s)
    return s

def parse_csv_list(text: str) -> List[str]:
    if not text:
        return []
    return [norm_name(x) for x in text.split(",") if norm_name(x)]

In [ ]:
# ISO epi weeks
def month_date_range(year: int, month: int):
    start = date(year, month, 1)
    end = date(year + (1 if month == 12 else 0), (1 if month == 12 else month + 1), 1)
    d = start
    while d < end:
        yield d
        d += timedelta(days=1)

def iso_weeks_in_month(year: int, month: int) -> List[Tuple[int, int, date]]:
    weeks = {}
    for d in month_date_range(year, month):
        iso_year, iso_week, iso_weekday = d.isocalendar()
        monday = d - timedelta(days=iso_weekday - 1)
        weeks[(iso_year, iso_week)] = monday
    return [(iy, iw, weeks[(iy, iw)]) for (iy, iw) in sorted(weeks.keys())]

In [ ]:
# Build skeleton rows
def resolve_geography(provinces: List[str], districts: List[str], seed: int = 42) -> List[Dict[str, str]]:
    rng = random.Random(seed)
    provinces = [p for p in map(norm_name, provinces or []) if p in PROVINCES]
    districts = [d for d in map(norm_name, districts or []) if d]

    pairs = []
    if districts:
        fallback_prov = provinces[0] if provinces else rng.choice(PROVINCES)
        for d in districts:
            prov = district_to_province.get(d.lower(), fallback_prov)
            pairs.append({"province": prov, "district": d})
        return pairs

    if provinces:
        for p in provinces:
            for d in province_to_districts[p]:
                pairs.append({"province": p, "district": d})
        return pairs

    # default sample if nothing provided
    p = rng.choice(PROVINCES)
    ds = province_to_districts[p]
    for d in rng.sample(ds, k=min(3, len(ds))):
        pairs.append({"province": p, "district": d})
    return pairs


def build_structure_rows(year: int, month: int, diseases: List[str], provinces: List[str], districts: List[str]) -> List[Dict[str, object]]:
    weeks = iso_weeks_in_month(year, month)
    geo_pairs = resolve_geography(provinces, districts)
    rows = []
    for gp in geo_pairs:
        for iso_year, epi_week, week_start in weeks:
            for disease in diseases:
                rows.append({
                    "disease": disease,
                    "province": gp["province"],
                    "district": gp["district"],
                    "year": year,
                    "month": month,
                    "iso_year": iso_year,
                    "epi_week": epi_week,
                    "week_start_date": week_start.isoformat(),
                })
    return rows

In [ ]:
# LLM prompt (CSV only)
SYSTEM_CSV_ONLY = """
You are generating synthetic public health surveillance data.

ABSOLUTE OUTPUT RULES:
- Output ONLY comma-delimited rows.
- Do NOT include headers.
- Do NOT include explanations.
- Do NOT include code fences.
- Do NOT include blank lines.
- No leading/trailing spaces.
- Output exactly the same number of rows as the input list (same order).

For each input row, output ONE corresponding CSV row with EXACTLY 10 fields:
disease,province,district,iso_year,epi_week,week_start_date,suspected_cases,cases_sent_to_lab,confirmed_cases,deaths

Constraints per row:
0 <= deaths <= confirmed_cases <= cases_sent_to_lab <= suspected_cases
All values must be integers.
""".strip()

def build_llm_prompt(struct_rows: List[Dict[str, object]], realism: str) -> str:
    simplified = [
        {
            "disease": r["disease"],
            "province": r["province"],
            "district": r["district"],
            "iso_year": r["iso_year"],
            "epi_week": r["epi_week"],
            "week_start_date": r["week_start_date"],
        }
        for r in struct_rows
    ]
    return f"""
Realism guidance:
{realism}

INPUT ROWS (same order must be preserved):
{json.dumps(simplified, ensure_ascii=False)}

Remember: output ONLY CSV rows, no header, no commentary.
""".strip()

In [ ]:
# Model calling (Ollama + HF)
def call_llm_csv(provider: str, model: str, struct_rows: List[Dict[str, object]], realism: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_CSV_ONLY},
        {"role": "user", "content": build_llm_prompt(struct_rows, realism)},
    ]

    if provider == "Ollama":
        try:
            resp = ollama.chat.completions.create(model=model, messages=messages)
            return (resp.choices[0].message.content or "").strip()
        except Exception as e:
            return f"ERROR_CALLING_MODEL: {type(e).__name__}: {e}"

    if provider == "HuggingFace":
        if not hf:
            return "ERROR_CALLING_MODEL: HuggingFace client not available (missing HUGGING_FACE_TOKEN)."
        try:
            r = hf.chat_completion(model=model, messages=messages, max_tokens=1500, temperature=0.2)
            return (r.choices[0].message.content or "").strip()
        except Exception:
            prompt = f"{SYSTEM_CSV_ONLY}\n\n{build_llm_prompt(struct_rows, realism)}"
            r = hf.text_generation(model=model, prompt=prompt, max_new_tokens=1500, temperature=0.2)
            return (r or "").strip()

    return "ERROR_CALLING_MODEL: Unknown provider"

In [ ]:
# CSV parsing
def extract_candidate_lines(text: str) -> List[str]:
    return [ln.strip() for ln in (text or "").splitlines() if ln.strip()]

def parse_csv_line(line: str):
    reader = csv.reader(io.StringIO(line))
    parts = next(reader, None)
    if not parts:
        return None

    parts = [p.strip() for p in parts]
    if len(parts) != 10:
        return None

    try:
        disease = parts[0].strip('"')
        province = parts[1].strip('"')
        district = parts[2].strip('"')
        iso_year = int(parts[3].strip('"'))
        epi_week = int(parts[4].strip('"'))
        week_start_date = parts[5].strip('"')
        suspected = int(parts[6].strip('"'))
        sent = int(parts[7].strip('"'))
        confirmed = int(parts[8].strip('"'))
        deaths = int(parts[9].strip('"'))

        if disease not in DISEASES:
            return None

        return (disease, province, district, iso_year, epi_week, week_start_date, suspected, sent, confirmed, deaths)
    except Exception:
        return None

def clamp_counts(suspected: int, sent: int, confirmed: int, deaths: int):
    suspected = max(0, int(suspected))
    sent = max(0, int(sent))
    confirmed = max(0, int(confirmed))
    deaths = max(0, int(deaths))
    sent = min(sent, suspected)
    confirmed = min(confirmed, sent)
    deaths = min(deaths, confirmed)
    return suspected, sent, confirmed, deaths

def parse_llm_output_to_df(llm_text: str, year: int, month: int) -> pd.DataFrame:
    lines = extract_candidate_lines(llm_text)
    parsed = []
    for ln in lines:
        row = parse_csv_line(ln)
        if row:
            parsed.append(row)

    if not parsed:
        raise ValueError("No valid CSV rows detected. Reduce rows or switch model.")

    out_rows = []
    for (disease, province, district, iso_year, epi_week, week_start_date, suspected, sent, confirmed, deaths) in parsed:
        suspected, sent, confirmed, deaths = clamp_counts(suspected, sent, confirmed, deaths)

        prov = norm_name(province)
        dist = norm_name(district)

        # Ensure province is populated correctly
        if not prov:
            prov = district_to_province.get(dist.lower(), "")

        out_rows.append({
            "disease": disease,
            "province": prov,
            "district": dist,
            "year": year,
            "month": month,
            "iso_year": int(iso_year),
            "epi_week": int(epi_week),
            "week_start_date": week_start_date,
            "suspected_cases": suspected,
            "cases_sent_to_lab": sent,
            "confirmed_cases": confirmed,
            "deaths": deaths,
        })

    df = pd.DataFrame(out_rows)
    df = df[COLUMNS]
    return df

In [ ]:
# Gradio callbacks
def update_model_dropdown(provider_name: str):
    models = MODEL_OPTIONS.get(provider_name, [])
    default = models[0] if models else None
    return gr.Dropdown(choices=models, value=default)

def preview_structure(year, month, diseases, provinces, districts_csv, limit_rows):
    year = int(year)
    month = int(month)
    diseases = diseases or []
    provinces = provinces or []
    districts = parse_csv_list(districts_csv)
    rows = build_structure_rows(year, month, diseases, provinces, districts)[:int(limit_rows)]
    return pd.DataFrame(rows)

def get_llm_output(year, month, diseases, provinces, districts_csv, provider_name, model_name, realism, limit_rows):
    year = int(year)
    month = int(month)
    diseases = diseases or []
    provinces = provinces or []
    districts = parse_csv_list(districts_csv)

    if not diseases:
        return "", "Select at least one disease.", pd.DataFrame()

    struct_rows = build_structure_rows(year, month, diseases, provinces, districts)[:int(limit_rows)]
    raw = call_llm_csv(provider_name, model_name, struct_rows, realism)

    # best-effort parse preview
    try:
        df = parse_llm_output_to_df(raw, year, month)
        status = f"Parsed rows: {len(df)} (expected {len(struct_rows)})"
        preview = df.head(25)
    except Exception as e:
        status = f"Parse warning: {e}"
        preview = pd.DataFrame()

    return raw, status, preview

def generate_csv(year, month, raw_llm_output):
    year = int(year)
    month = int(month)
    df = parse_llm_output_to_df(raw_llm_output, year, month)
    filename = f"zambia_health_epi_{year}_{month:02d}.csv"
    df.to_csv(filename, index=False)
    return df.head(50), filename

In [ ]:
# Gradio UI
with gr.Blocks(title="Zambia Health Surveillance Synthetic Data (Ollama + HF)") as ui:
    gr.Markdown(
        "# Zambia Health Surveillance Synthetic Data Generator\n"
        "Workflow:\n"
        "1) Preview the **skeleton structure**\n"
        "2) Get the **raw LLM output** (CSV rows only)\n"
        "3) Generate the **CSV** from that raw output\n"
    )

    with gr.Row():
        diseases = gr.CheckboxGroup(choices=DISEASES, value=["Malaria", "Cholera"], label="Diseases / conditions")
        year = gr.Number(value=2026, precision=0, label="Year")
        month = gr.Dropdown(choices=[str(i) for i in range(1, 13)], value="3", label="Month (1-12)")

    gr.Markdown("### Geography")
    with gr.Row():
        provinces = gr.CheckboxGroup(choices=PROVINCES, value=["Lusaka"], label="Provinces (optional)")
        districts_csv = gr.Textbox(
            label="Districts override (optional, comma-separated)",
            value="",
            placeholder="If provided, province is inferred from district-to-province mapping."
        )

    gr.Markdown("### Model Provider")
    with gr.Row():
        provider_name = gr.Dropdown(choices=["Ollama", "HuggingFace"], value="HuggingFace", label="Provider")
        model_name = gr.Dropdown(choices=MODEL_OPTIONS["HuggingFace"], value=MODEL_OPTIONS["HuggingFace"][0], label="Model")

    provider_name.change(
        fn=update_model_dropdown,
        inputs=[provider_name],
        outputs=[model_name],
    )

    realism = gr.Textbox(
        label="Realism guidance",
        lines=4,
        value="Anthrax is rare. Cholera can spike in outbreaks. Malaria may be higher in rainy season. Keep deaths low relative to confirmed."
    )

    limit_rows = gr.Number(value=30, precision=0, label="How many skeleton rows to request from the LLM (debug/safety)")

    with gr.Row():
        btn_preview = gr.Button("Preview Structure")
        btn_llm = gr.Button("Get LLM Output (raw)")
        btn_csv = gr.Button("Generate CSV")

    structure_table = gr.Dataframe(label="Structure preview (skeleton rows)")
    llm_raw = gr.Textbox(label="Raw LLM output (CSV rows only)", lines=16)
    status = gr.Textbox(label="Status / validation")
    parsed_table = gr.Dataframe(label="Parsed preview (first 25 rows)")
    df_preview = gr.Dataframe(label="Final DataFrame preview (first 50 rows)")
    download = gr.File(label="Download CSV")

    btn_preview.click(
        fn=preview_structure,
        inputs=[year, month, diseases, provinces, districts_csv, limit_rows],
        outputs=[structure_table]
    )

    btn_llm.click(
        fn=get_llm_output,
        inputs=[year, month, diseases, provinces, districts_csv, provider_name, model_name, realism, limit_rows],
        outputs=[llm_raw, status, parsed_table]
    )

    btn_csv.click(
        fn=generate_csv,
        inputs=[year, month, llm_raw],
        outputs=[df_preview, download]
    )

ui.launch(inbrowser=True)